Climate Health Vulnerability Cluster Analysis Code

In [1]:
shiny_code = r'''
library(shiny)
library(tidyverse)
library(plotly)
library(DT)
library(ggrepel)

setwd("C:/Users/xx/Documents/yy")

files <- c("Integrated_Data1.csv")

dfs <- lapply(files, function(f) {
  df <- read.csv(f, stringsAsFactors = FALSE)
  names(df) <- names(df) %>%
    tolower() %>%
    gsub("[^a-z0-9_]", "_", .) %>%
    gsub("_+", "_", .)
  if (!"census_tract" %in% names(df)) stop(paste("Missing census_tract in", f))
  df <- df %>% mutate(census_tract = as.factor(census_tract))
  char_cols <- sapply(df, is.character)
  if (any(char_cols)) {
    df[char_cols] <- lapply(df[char_cols], function(x) as.numeric(gsub("[^0-9.-]", "", x)))
  }
  df %>%
    select(census_tract, where(is.numeric)) %>%
    mutate(across(where(is.numeric), ~ ifelse(is.infinite(.x), NA, .x))) %>%
    drop_na()
})

df_all <- bind_rows(dfs) %>%
  group_by(census_tract) %>%
  summarise(across(where(is.numeric), mean, na.rm = TRUE), .groups = "drop")

num_cols <- setdiff(names(df_all), "census_tract")
na_pct <- sapply(df_all[num_cols], function(x) mean(is.na(x)))
vars_keep <- names(na_pct[na_pct <= 0.5])
vars_keep <- vars_keep[sapply(df_all[vars_keep], function(x) sd(x, na.rm = TRUE) > 0)]
if (length(vars_keep) < 2) stop("Not enough valid variables for PCA/clustering. Check your data.")

df_model <- df_all %>%
  mutate(across(all_of(vars_keep), ~ ifelse(is.infinite(.x), NA, .x))) %>%
  drop_na(all_of(vars_keep)) %>%
  mutate(average_risk = rowMeans(across(all_of(vars_keep)), na.rm = TRUE))

num_data <- scale(df_model[, vars_keep, drop = FALSE])
pca_res <- prcomp(num_data, center = TRUE, scale. = TRUE)
km_res <- kmeans(num_data, centers = 4, nstart = 25)
df_model$cluster <- factor(km_res$cluster)

df_pca <- data.frame(
  PC1 = pca_res$x[, 1],
  PC2 = pca_res$x[, 2],
  census_tract = df_model$census_tract,
  cluster = df_model$cluster
)

pretty_label <- function(x) {
  x %>%
    gsub("_+", " ", .) %>%
    gsub("\\b([a-z])", "\\U\\1", ., perl = TRUE) %>%
    str_trunc(40, ellipsis = "...") %>%
    str_squish()
}

var_choices <- setNames(vars_keep, pretty_label(vars_keep))



In [ ]:
ui <- fluidPage(
  titlePanel("Climate Health Vulnerability Dashboard"),
  sidebarLayout(
    sidebarPanel(
      selectInput("cluster_view", "Select Data View",
                  choices = c("All Census Tracts", paste("Cluster", 1:4))),
      uiOutput("variable_ui"),
      sliderInput("tract_range", "Select Census Tract Range",
                  min = 1, max = nrow(df_model), value = c(1, min(30, nrow(df_model)))),
      checkboxInput("show_labels", "Show Census Tract Labels", TRUE)
    ),
    mainPanel(
      tabsetPanel(
        tabPanel("Vulnerability Heatmap", plotlyOutput("heatmapPlot", height = "650px")),
        tabPanel("Top Vulnerable Tracts", DTOutput("topTable")),
        tabPanel("PCA + Clustering", plotlyOutput("pcaPlot", height = "650px")),
        tabPanel("Cluster Summary", DTOutput("clusterTable"))
      )
    )
  )
)

server <- function(input, output, session) {
  output$variable_ui <- renderUI({
    selectInput("selected_vars", "Select Variables for Analysis",
                choices = var_choices, selected = vars_keep, multiple = TRUE)
  })

  data_view <- reactive({
    df <- df_model
    if (input$cluster_view != "All Census Tracts") {
      cluster_num <- as.numeric(gsub("Cluster ", "", input$cluster_view))
      df <- df %>% filter(cluster == cluster_num)
    }
    df
  })

  tract_filtered <- reactive({
    df <- data_view()
    req(nrow(df) > 0)
    min_idx <- max(1, min(input$tract_range[1], nrow(df)))
    max_idx <- max(1, min(input$tract_range[2], nrow(df)))
    if (min_idx > max_idx) {
      min_idx <- 1
      max_idx <- min(30, nrow(df))
    }
    df %>% slice(min_idx:max_idx)
  })

  output$heatmapPlot <- renderPlotly({
    req(input$selected_vars)
    df <- tract_filtered()
    req(nrow(df) > 0)
    df_long <- df %>%
      select(census_tract, cluster, all_of(input$selected_vars)) %>%
      pivot_longer(cols = all_of(input$selected_vars), names_to = "Variable", values_to = "Value") %>%
      mutate(Variable = pretty_label(Variable))
    p <- ggplot(df_long, aes(census_tract, Variable, fill = Value)) +
      geom_tile(color = "white") +
      scale_fill_gradient(low = "lightyellow", high = "red") +
      labs(title = "Census Tract Vulnerability Heatmap", x = "Census Tract", y = "Indicator", fill = "Value") +
      theme_minimal() +
      theme(axis.text.x = element_text(angle = 45, hjust = 1))
    ggplotly(p)
  })

  output$topTable <- renderDT({
    df_ranked <- df_model %>%
      arrange(desc(average_risk)) %>%
      select(census_tract, average_risk, cluster, all_of(vars_keep))
    colnames(df_ranked)[-c(1,2,3)] <- pretty_label(colnames(df_ranked)[-c(1,2,3)])
    datatable(df_ranked, options = list(pageLength = 10, lengthMenu = c(10, 25, 50, 100), scrollX = TRUE))
  })

  

In [ ]:
output$pcaPlot <- renderPlotly({
    df <- df_pca
    if (input$cluster_view != "All Census Tracts") {
      cluster_num <- as.numeric(gsub("Cluster ", "", input$cluster_view))
      df <- df %>% filter(cluster == cluster_num)
    }
    p <- ggplot(df, aes(PC1, PC2, color = cluster, label = census_tract)) +
      geom_point(size = 3, alpha = 0.8) +
      labs(title = "PCA + K Means Clustering", color = "Cluster") +
      theme_minimal()
    if (isTRUE(input$show_labels)) p <- p + geom_text_repel(size = 3)
    ggplotly(p)
  })

  

In [ ]:
output$clusterTable <- renderDT({
    req(input$selected_vars)
    summary_tbl <- df_model %>%
      group_by(cluster) %>%
      summarise(
        tracts = n(),
        mean_risk = mean(average_risk, na.rm = TRUE),
        sd_risk = sd(average_risk, na.rm = TRUE),
        across(all_of(input$selected_vars), mean, .names = "mean_{.col}"),
        .groups = "drop"
      )
    summary_names <- names(summary_tbl)
    summary_names[grep("^mean_", summary_names)] <- pretty_label(gsub("^mean_", "", summary_names[grep("^mean_", summary_names)]))
    names(summary_tbl) <- summary_names
    datatable(summary_tbl, options = list(scrollX = TRUE))
  })
}

shinyApp(ui, server)
'''